# Lab 1: Data and Embeddings

This notebook introduces the dataset and illustrates how the raw inputs are structured. Notebooks 2 and 3 are fully self-contained and do not depend on this one.

## Background

The data come from Amazon.com and cover toy products observed between March 2023 and January 2024. For each product, we observe monthly snapshots of the sales rank and the buy-box price, together with product descriptions and images. The dataset contains approximately 7,000 unique products tracked over 12 four-week periods.

The two central variables are defined as follows. Let $i$ index products and $t$ index time periods.

**Quantity signal.** Amazon does not publish unit sales directly. We use the sales rank as a proxy. Higher sales rank means fewer units sold, so we invert it. The quantity signal is
$$Q_{it} = \log\!\left(\frac{1}{\text{sales rank}_{it}}\right).$$
A higher value of $Q_{it}$ corresponds to a product selling more units relative to competitors in the category.

**Price signal.** The buy-box price is the price shown to buyers considering immediate purchase. We transform it as
$$P_{it} = \log(\text{buy-box price}_{it}).$$
Working in logs means that regression coefficients can be interpreted as approximate elasticities.

## Embeddings

Each product's text description and image are passed through transformer-based models (RoBERTa for text, BEiT for images) and the resulting representations are compressed to 256 dimensions. These embedding vectors, denoted $X_i^e \in \mathbb{R}^{256}$, serve as rich product-level controls that capture quality, style, and category information that is not available in structured form.

Because the raw embedding space is high-dimensional, we summarize it in two ways for use in regression:
- **PCA projections** retain the directions of maximum variance.
- **Cluster similarities** measure cosine proximity to $K = 5$ cluster centroids obtained by k-means, giving a compressed vector $X_i^{\rm sim} \in \mathbb{R}^K$.

These summaries play a central role in the causal estimation in Lab 3.

In [ ]:
import datasets
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import normalize

palette = sns.color_palette("colorblind")

## Loading the Dataset

The dataset is hosted on HuggingFace. It is split into a training set and a test set. We use this split throughout all three labs.

In [ ]:
ds = datasets.load_dataset("janteichertkluge/demand-analysis-repro")
print(ds)

ds_train = ds["train"]
ds_test  = ds["test"]
print(f"\nTrain: {len(ds_train):,} observations")
print(f"Test:  {len(ds_test):,} observations")

In [ ]:
all_cols = ds_train.column_names
emb_cols = sorted([c for c in all_cols if c.startswith("emb_")])
meta_cols = [c for c in all_cols if c not in emb_cols]

print(f"Embedding dimensions:  {len(emb_cols)}")
print(f"Structured variables:  {len(meta_cols)}")
print("\nStructured columns:")
for c in meta_cols:
    print(" ", c)

In [ ]:
df_train = ds_train.to_pandas()
df_test  = ds_test.to_pandas()

# Apply log transformations consistent with the paper
for df in [df_train, df_test]:
    df["Q_t"] = np.log(1.0 / df["SALES_RANK"].clip(lower=1))
    df["P_t"] = np.log(df["BUYBOX_PRICE"].clip(lower=0.01))

df_train[["ASIN", "date", "Q_t", "P_t"]].head()

## Descriptive Statistics

We look at the distributions of the quantity and price signals and the number of products per subcategory.

In [ ]:
df_train[["Q_t", "P_t", "REVIEW_COUNT", "RATING"]].describe().round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df_train["Q_t"].dropna(), bins=80, color=palette[0], alpha=0.8, edgecolor="none")
axes[0].set_xlabel("$Q_{it} = \\log(1/\\mathrm{sales\\, rank})$", fontsize=11)
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of the Quantity Signal")
axes[0].grid(axis="y", alpha=0.3)

axes[1].hist(df_train["P_t"].dropna(), bins=80, color=palette[1], alpha=0.8, edgecolor="none")
axes[1].set_xlabel("$P_{it} = \\log(\\mathrm{buy\\text{-}box\\, price})$", fontsize=11)
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of the Price Signal")
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
if "subcat_aggregated" in df_train.columns:
    print("Products per subcategory (training set):")
    print(df_train.groupby("subcat_aggregated")["ASIN"].nunique().sort_values(ascending=False).to_string())

## Time Series Overview

To get a feel for the data, we plot the median quantity and price signal over time. The panel is unbalanced, so these medians mix products entering and leaving the sample.

In [ ]:
df_train["date_dt"] = pd.to_datetime(df_train["date"])
monthly = df_train.groupby("date_dt")[["Q_t", "P_t"]].median()

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(monthly.index, monthly["Q_t"], color=palette[0], linewidth=2)
axes[0].set_ylabel("Median $Q_{it}$", fontsize=11)
axes[0].set_title("Median Quantity Signal over Time (training set)")
axes[0].grid(alpha=0.3)

axes[1].plot(monthly.index, monthly["P_t"], color=palette[1], linewidth=2)
axes[1].set_ylabel("Median $P_{it}$", fontsize=11)
axes[1].set_title("Median Price Signal over Time (training set)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Embedding Structure

The 256-dimensional embedding vectors are centered (subtract training mean) and L2-normalized before any further processing. We then apply PCA to visualize the main axes of variation.

Normalizing to unit length ensures that all products are compared by direction rather than magnitude, which is standard practice when working with transformer embeddings.

In [ ]:
emb_tr = df_train[emb_cols].values.astype(float)
emb_te = df_test[emb_cols].values.astype(float)

global_mean = emb_tr.mean(axis=0)
emb_tr_n = normalize(emb_tr - global_mean, axis=1)
emb_te_n = normalize(emb_te - global_mean, axis=1)

pca10 = PCA(n_components=10, random_state=42)
pca10.fit(emb_tr_n)

print("Variance explained by PCA components:")
for i, ev in enumerate(pca10.explained_variance_ratio_):
    cumulative = pca10.explained_variance_ratio_[:i+1].sum()
    print(f"  PC{i+1:2d}:  {ev:.3f}  (cumulative: {cumulative:.3f})")

In [ ]:
pca2 = PCA(n_components=2, random_state=42)
coords = pca2.fit_transform(emb_tr_n)

if "subcat_aggregated" in df_train.columns:
    cats = df_train["subcat_aggregated"].astype("category")
    codes = cats.cat.codes.values
else:
    cats  = pd.Categorical(["all"] * len(coords))
    codes = np.zeros(len(coords), dtype=int)

fig, ax = plt.subplots(figsize=(8, 6))
for i, cat in enumerate(cats.cat.categories):
    mask = codes == i
    ax.scatter(coords[mask, 0], coords[mask, 1],
               label=cat, alpha=0.3, s=6, color=palette[i % len(palette)])
ax.set_xlabel("First principal component", fontsize=11)
ax.set_ylabel("Second principal component", fontsize=11)
ax.set_title("Text Embeddings Projected onto the First Two PCs (training set)")
ax.legend(markerscale=3, fontsize=8, loc="best")
plt.tight_layout()
plt.show()

## K-Means Clustering

We fit a k-means model with $K = 5$ clusters to the normalized training embeddings. The resulting centroids define the similarity variables $X_i^{\rm sim}$ used in the regression models. Each similarity score is the cosine similarity between a product's embedding and the corresponding cluster centroid.

In [ ]:
km = KMeans(n_clusters=5, random_state=42, n_init=50)
labels = km.fit_predict(emb_tr_n)

fig, ax = plt.subplots(figsize=(8, 6))
for k in range(5):
    mask = labels == k
    ax.scatter(coords[mask, 0], coords[mask, 1],
               label=f"Cluster {k+1}", alpha=0.3, s=6, color=palette[k])
ax.set_xlabel("First principal component", fontsize=11)
ax.set_ylabel("Second principal component", fontsize=11)
ax.set_title("K-Means Clusters in the Embedding Space (training set)")
ax.legend(markerscale=3)
plt.tight_layout()
plt.show()

print("Products per cluster (training set):")
for k in range(5):
    print(f"  Cluster {k+1}: {(labels == k).sum():,}")